## Imports

In [32]:
import pandas as pd

## Read in NORC Data

In [33]:
state_df = pd.read_csv('../data/raw-data/NORC/AP_NORC_survey.csv')
state_df.shape

(139938, 466)

## Analyze Value Counts for Variables of Interest
- MAGA, PARTY, RELIG, EDUC, SIZEPLACE (urban/rural/suburban status)

In [34]:
state_df.MAGA.value_counts(dropna=False, normalize=True)

MAGA
(2) No                                          0.620611
(1) Yes                                         0.371529
(99) DON'T KNOW/SKIPPED ON WEB/REFUSED (VOL)    0.007861
Name: proportion, dtype: float64

In [35]:
state_df.PARTY.value_counts(dropna=False, normalize=True)

PARTY
(1) Democrat                                    0.365526
(2) Republican                                  0.346596
(3) Neither                                     0.287420
(99) DON'T KNOW/SKIPPED ON WEB/REFUSED (VOL)    0.000457
Name: proportion, dtype: float64

In [36]:
state_df.RELIG.value_counts(dropna=False, normalize=True)

RELIG
(8) None                                        0.228630
(1) Protestant                                  0.200582
(2) Catholic                                    0.184382
(4) Other Christian                             0.180065
(7) Something else                              0.078020
(88) REMOVED FOR DISCLOSURE RISK                0.071767
(5) Jewish                                      0.022681
(3) Mormon                                      0.018644
(6) Muslim                                      0.011669
(99) DON'T KNOW/SKIPPED ON WEB/REFUSED (VOL)    0.003559
Name: proportion, dtype: float64

In [37]:
state_df.SIZEPLACE.value_counts(dropna=False, normalize=True)

SIZEPLACE
(2) Suburban                                    0.408388
(1) Urban                                       0.220212
(4) Rural                                       0.185975
(3) Small town                                  0.181688
(99) DON'T KNOW/SKIPPED ON WEB/REFUSED (VOL)    0.003737
Name: proportion, dtype: float64

In [38]:
state_df.EDUC.value_counts(dropna=False, normalize=True)

EDUC
(2) Some college/assoc. degree                  0.335634
(1) High school or less                         0.262073
(3) College graduate                            0.232374
(4) Postgraduate study                          0.169304
(99) DON'T KNOW/SKIPPED ON WEB/REFUSED (VOL)    0.000615
Name: proportion, dtype: float64

## Read in SAVE Act Tables

In [39]:
save_act_df = pd.read_excel("../data/raw-data/SAVEAct_tables.xlsx")
save_act_df

,State,(15 years and older) whose names do not match their birth certificate (last name change or hyphenation),Female citizens 15 and older who are married or have been married,Estimated number of female citizens whose names do not match their birth certificate (last name change only)
0,Alabama,1153766.0,1373531,1085090
1,Alaska,155961.0,185668,146678
2,Arizona,1527920.0,1818952,1436972
3,Arkansas,697813.0,830730,656277
4,California,7033264.0,8372933,6614617
5,Colorado,1268345.0,1509934,1192848
6,Connecticut,747439.0,889808,702948
7,Delaware,440667.0,524603,414436
8,Florida,4757384.0,5663552,4474206
9,Georgia,2214380.0,2636167,2082572


In [40]:
# Clean State Column from NPORS to join
state_df['P_STATE'] = state_df['P_STATE'].str.replace(r'^\([A-Z]{2}\)\s*', '', regex=True)
state_df['P_STATE'].value_counts()

P_STATE
Texas             5778
Florida           5670
California        5644
New York          5115
Arizona           4875
Georgia           4762
Ohio              4675
Pennsylvania      4629
Wisconsin         4567
North Carolina    4417
Michigan          4404
Maryland          4329
Nevada            4099
Minnesota         4016
Colorado          3985
Virginia          3303
Illinois          3224
Indiana           3180
Tennessee         3012
Louisiana         2924
South Carolina    2890
Missouri          2843
Washington        2842
Massachusetts     2801
Kentucky          2714
New Jersey        2687
Oregon            2672
Connecticut       2652
Arkansas          2646
Utah              2401
Iowa              2200
New Mexico        2189
Maine             2136
Nebraska          2051
New Hampshire     2029
Kansas            2014
Alaska            1430
Montana           1375
West Virginia     1181
Alabama           1133
Oklahoma          1108
Mississippi       1065
South Dakota       885
Wyo

In [41]:
merged = state_df.merge(save_act_df, how='left',left_on='P_STATE',right_on='State')

## Create Flag Columns for Scatterplots

In [42]:
women_df = merged[merged['GENDER'].eq('(2) Women')].copy()

women_df['is_maga'] = women_df['MAGA'].eq('(1) Yes').astype(int)

women_df['is_any_religion'] = ~women_df['RELIG'].isin([
    '(8) None',
    '(99) DON\'T KNOW/SKIPPED ON WEB/REFUSED (VOL)',
    '(88) REMOVED FOR DISCLOSURE RISK'
])

women_df['is_any_religion'] = women_df['is_any_religion'].astype(int)

women_df['is_christian'] = women_df['RELIG'].isin([
    '(1) Protestant',
    '(2) Catholic',
    '(3) Mormon',
    '(4) Other Christian'
]).astype(int)

women_df['is_college_ed'] = women_df['EDUC'].isin(["(3) College graduate",'(4) Postgraduate study']).astype(int)
women_df['less_than_bachelors'] = (~women_df['EDUC'].isin(["(3) College graduate",'(4) Postgraduate study'])).astype(int)

women_df['is_rural'] = women_df['SIZEPLACE'].isin(['(4) Rural','(3) Small town']).astype(int)
women_df['is_urban'] = women_df['SIZEPLACE'].isin(['(1) Urban']).astype(int)
women_df['is_suburban'] = women_df['SIZEPLACE'].isin(['(2) Suburban']).astype(int)

state_grouped = (
    women_df.groupby('P_STATE')
    .agg(
        pct_maga=('is_maga', 'mean'),
        pct_christian=('is_christian', 'mean'),
        pct_college_ed=('is_college_ed', 'mean'),
        pct_any_religion = ('is_any_religion','mean'),
        pct_rural=('is_rural', 'mean'),
        pct_urban = ('is_urban','mean'),
        pct_suburban = ('is_suburban','mean'),
        pct_less_than_bachelors = ('less_than_bachelors','mean'),
        n=('P_STATE', 'size')
    )
    .reset_index()
)

cols = ['pct_maga', 'pct_christian', 'pct_any_religion','pct_less_than_bachelors','pct_college_ed','pct_rural','pct_urban','pct_suburban']
state_grouped[cols] = state_grouped[cols] * 100

In [43]:
import plotly.express as px

outcomes = ['pct_christian',
    'pct_less_than_bachelors','pct_college_ed',
    'pct_rural','pct_urban','pct_suburban']

for y_var in outcomes:

    fig = px.scatter(
        state_grouped,
        x='pct_maga',
        y=y_var,
        color='pct_maga',
        color_continuous_scale="Bluered",
        size='n',
        hover_name='P_STATE',
        title=f'{y_var} vs Percentage of Women Who Identify as MAGA'
    )

    fig.update_traces(
        text=state_grouped['P_STATE'],
        textposition='top center'
    )

    fig.show()